# Generating All Subsets
This notebook demonstrates a clear backtracking template for producing every subset of a given sequence.

In [2]:
from dataclasses import dataclass, field
from typing import List, Sequence, Tuple, TypeVar

T = TypeVar("T")


@dataclass
class SubsetState:
    index: int = 0
    path: List[T] = field(default_factory=list)
    checkpoints: List[Tuple[int, int]] = field(default_factory=list)


def find_subsets(items: Sequence[T]) -> List[List[T]]:
    """Return every distinct subset of `items` while preserving the original order."""
    values = list(items)
    result: List[List[T]] = []
    seen: set[Tuple[T, ...]] = set()
    state = SubsetState()

    def goal(current: SubsetState) -> bool:
        return current.index >= len(values)

    def record(current: SubsetState) -> None:
        snapshot = tuple(current.path)
        if snapshot not in seen:
            seen.add(snapshot)
            result.append(list(snapshot))

    def choices(current: SubsetState) -> List[bool]:
        return [] if goal(current) else [True, False]

    def valid(_: SubsetState, __: bool) -> bool:
        return True

    def apply(current: SubsetState, choice: bool) -> None:
        current.checkpoints.append((current.index, len(current.path)))
        if not choice:
            if current.index < len(values):
                value = values[current.index]
                next_index = current.index
                while next_index < len(values) and values[next_index] == value:
                    next_index += 1
                current.index = next_index
            return

        current.path.append(values[current.index])
        current.index += 1

    def undo(current: SubsetState, choice: bool) -> None:
        previous_index, previous_path_len = current.checkpoints.pop()
        current.index = previous_index
        while len(current.path) > previous_path_len:
            current.path.pop()

    def backtrack(current: SubsetState) -> bool:
        if goal(current):
            record(current)
            return False
        for choice in choices(current):
            if valid(current, choice):
                apply(current, choice)
                backtrack(current)
                undo(current, choice)
        return False

    backtrack(state)
    return result


In [3]:
find_subsets([1, 2, 2])  # Example usage

[[1, 2, 2], [1, 2], [1], [2, 2], [2], []]

In [ ]:
from dataclasses import dataclass, field
from typing import List, Sequence, TypeVar

T = TypeVar("T")


@dataclass
class SubsetState:
    index: int = 0
    path: List[T] = field(default_factory=list)


def find_subsets(items: Sequence[T]) -> List[List[T]]:
    """Return every subset of `items` while preserving the original order."""
    result: List[List[T]] = []
    state = SubsetState()

    def goal(current: SubsetState) -> bool:
        return current.index == len(items)

    def record(current: SubsetState) -> None:
        result.append(current.path.copy())

    def choices(current: SubsetState) -> List[bool]:
        return [] if current.index == len(items) else [True, False]

    def valid(_: SubsetState, __: bool) -> bool:
        return True

    def apply(current: SubsetState, choice: bool) -> None:
        if choice:
            current.path.append(items[current.index])
        current.index += 1

    def undo(current: SubsetState, choice: bool) -> None:
        current.index -= 1
        if choice:
            current.path.pop()
    def backtrack(current: SubsetState) -> bool:
        if goal(current):
            record(current)
            return False
        for choice in choices(current):
            if valid(current, choice):
                apply(current, choice)
                if backtrack(current):
                    return True
                undo(current, choice)
        return False

    backtrack(state)
    return result
